In [195]:

import operator
from typing import TypedDict, Annotated

from dotenv import load_dotenv
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

load_dotenv()


True

In [ ]:

class EassyEvaluationState(TypedDict):
    essay: str
    cot_feedback: str
    doa_feedback: str
    language_feedback: str
    individual_score: Annotated[list[float], operator.add]

    final_feedback: str
    final_average_score: float

In [197]:
class OutputParser(BaseModel):
    feedback: str = Field(description="Feedback returned by LLM in string format")
    score: float = Field(description="Score out of 10 returned by LLM in float format")

In [198]:
def eval_and_return_structured_reply(prompt: str) -> OutputParser:
    llm = ChatGroq(model="qwen/qwen3-32b", temperature=0)
    structured_llm = llm.with_structured_output(OutputParser)

    llm_output = structured_llm.invoke(prompt)
    return llm_output


In [199]:
def eval_and_return_reply(prompt: str) -> AIMessage:
    llm = ChatGroq(model="qwen/qwen3-32b", temperature=0)
    llm_output = llm.invoke(prompt)
    return llm_output

In [ ]:
def get_prompt(parameter: str, essay_content: str) -> str:
    return (
        f"Evaluate the essay content given below on the parameter of {parameter} and return 2 things "
        f"1. Feedback in string format "
        f"2. Score out of 10 in float \n \n"
        f"Essay : \n "
        f"{essay_content}")


In [ ]:
def cot_eval(state: EassyEvaluationState) -> TypedDict:
    result = eval_and_return_structured_reply(get_prompt("clarity of thoughts", state["essay"]))
    return {"cot_feedback": result.feedback, "individual_score": [result.score]}


In [202]:
def doa_eval(state: EassyEvaluationState) -> TypedDict:
    result = eval_and_return_structured_reply(get_prompt("depth of analysis", state["essay"]))
    return {"doa_feedback": result.feedback, "individual_score": [result.score]}

In [203]:
def language_eval(state: EassyEvaluationState) -> TypedDict:
    result = eval_and_return_structured_reply(get_prompt("language", state["essay"]))
    return {"language_feedback": result.feedback, "individual_score": [result.score]}

In [204]:
def final_eval(state: EassyEvaluationState) -> TypedDict:
    feedback = f"COT Feedback : {state['cot_feedback']} \n DOA Feedback : {state['doa_feedback']} \n Language Feedback : {state['language_feedback']}"

    prompt = (f"Based on the combined feedback given below, give me a final feedback \n"
              f"Combined Feedback : \n {feedback}")
    result = eval_and_return_reply(prompt).content

    return {"final_feedback": result,
            "final_average_score": sum(state["individual_score"]) / len(state["individual_score"])}

In [ ]:
graph = StateGraph(EassyEvaluationState)

graph.add_node("cot_eval", cot_eval)
graph.add_node("doa_eval", doa_eval)
graph.add_node("language_eval", language_eval)
graph.add_node("final_eval", final_eval)

graph.add_edge(START, "cot_eval")
graph.add_edge(START, "doa_eval")
graph.add_edge(START, "language_eval")
graph.add_edge("cot_eval", "final_eval")
graph.add_edge("doa_eval", "final_eval")
graph.add_edge("language_eval", "final_eval")
graph.add_edge("final_eval", END)

workflow = graph.compile()



In [206]:
initial_state = {
    "essay": "The essay provides a comprehensive analysis of the topic, demonstrating a clear understanding of the subject matter. The arguments are well-structured and supported by relevant evidence, showcasing depth of analysis. The language used is articulate and engaging, effectively conveying the writer's ideas. Overall, the essay excels in clarity of thoughts, depth of analysis, and language, making it a compelling read."}

workflow.invoke(initial_state)

{'essay': "The essay provides a comprehensive analysis of the topic, demonstrating a clear understanding of the subject matter. The arguments are well-structured and supported by relevant evidence, showcasing depth of analysis. The language used is articulate and engaging, effectively conveying the writer's ideas. Overall, the essay excels in clarity of thoughts, depth of analysis, and language, making it a compelling read.",
 'cot_feedback': 'The essay demonstrates excellent clarity of thoughts with a logical flow, well-structured arguments, and precise language. Each idea is presented coherently, supported by relevant evidence, and transitions smoothly to the next point. The writer effectively communicates complex ideas in an accessible manner, ensuring the reader can follow the analysis without confusion.',
 'doa_feedback': 'The essay discusses the importance of depth of analysis but fails to demonstrate it with specific examples or critical examination of the topic. The content is 